In [0]:
CREATE OR REPLACE TABLE mini_project.gold.datacube_kpi AS
WITH payroll_monthly AS (
  SELECT 
    p.company_id,
    p.company_name,
    p.department_id,
    p.department_name,
    p.cost_center,
    date_trunc('month', p.pay_period_end) AS period_month,
    year(p.pay_period_end) AS year,
    month(p.pay_period_end) AS month,
    quarter(p.pay_period_end) AS quarter,
    -- Payroll KPIs
    COUNT(DISTINCT p.employee_id) AS employee_count,
    SUM(p.gross_salary) AS total_gross_salary,
    SUM(p.bonus) AS total_bonus,
    SUM(p.overtime_pay) AS total_overtime,
    SUM(p.commission) AS total_commission,
    SUM(p.allowances) AS total_allowances,
    SUM(p.total_compensation) AS total_compensation,
    SUM(p.total_deductions) AS total_deductions,
    SUM(p.tax_deduction) AS total_tax_deduction,
    SUM(p.social_security) AS total_social_security,
    SUM(p.health_insurance) AS total_health_insurance,
    SUM(p.retirement_contribution) AS total_retirement,
    SUM(p.other_deductions) AS total_other_deductions,
    SUM(p.net_salary) AS total_net_salary,
    AVG(p.gross_salary) AS avg_gross_salary,
    AVG(p.total_compensation) AS avg_total_compensation,
    AVG(p.overtime_to_base_ratio) AS avg_overtime_ratio
  FROM mini_project.gold.fact_payroll p
  GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
),
gl_monthly AS (
  SELECT 
    gl.company_id,
    gl.company_name,
    gl.department_id,
    gl.department_name,
    gl.cost_center,
    date_trunc('month', gl.entry_date) AS period_month,
    year(gl.entry_date) AS year,
    month(gl.entry_date) AS month,
    quarter(gl.entry_date) AS quarter,
    -- GL KPIs by P&L section - use credit_amount for revenue (gross sales)
    SUM(CASE WHEN gl.pl_section = 'Revenue' THEN gl.credit_amount ELSE 0 END) AS total_revenue,
    SUM(CASE WHEN gl.pl_section = 'Cost of Sales' THEN ABS(gl.net_amount) ELSE 0 END) AS total_cogs,
    SUM(CASE WHEN gl.pl_section = 'Personnel Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_personnel_expense,
    SUM(CASE WHEN gl.pl_section = 'Occupancy Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_occupancy_expense,
    SUM(CASE WHEN gl.pl_section = 'Technology Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_technology_expense,
    SUM(CASE WHEN gl.pl_section = 'Marketing Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_marketing_expense,
    SUM(CASE WHEN gl.pl_section = 'Travel & Entertainment' THEN ABS(gl.net_amount) ELSE 0 END) AS total_travel_expense,
    SUM(CASE WHEN gl.pl_section = 'Professional Services' THEN ABS(gl.net_amount) ELSE 0 END) AS total_professional_services,
    SUM(CASE WHEN gl.pl_section = 'Communication Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_communication_expense,
    SUM(CASE WHEN gl.pl_section = 'Financial Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_financial_expense,
    SUM(CASE WHEN gl.pl_section = 'Depreciation & Amortization' THEN ABS(gl.net_amount) ELSE 0 END) AS total_depreciation,
    SUM(CASE WHEN gl.pl_section = 'Other Operating Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_other_opex,
    SUM(CASE WHEN gl.pl_section = 'R&D Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_rnd_expense,
    SUM(CASE WHEN gl.pl_section = 'Tax Expense' THEN ABS(gl.net_amount) ELSE 0 END) AS total_tax_expense,
    SUM(CASE WHEN gl.pl_section != 'Revenue' THEN ABS(gl.net_amount) ELSE 0 END) AS total_expenses,
    COUNT(DISTINCT gl.gl_id) AS transaction_count
  FROM mini_project.gold.fact_general_ledger gl
  GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
)
SELECT 
  -- Dimension keys
  COALESCE(p.company_id, g.company_id) AS company_id,
  COALESCE(p.company_name, g.company_name) AS company_name,
  COALESCE(p.department_id, g.department_id) AS department_id,
  COALESCE(p.department_name, g.department_name) AS department_name,
  COALESCE(p.cost_center, g.cost_center) AS cost_center,
  COALESCE(p.period_month, g.period_month) AS period_month,
  COALESCE(p.year, g.year) AS year,
  COALESCE(p.month, g.month) AS month,
  COALESCE(p.quarter, g.quarter) AS quarter,
  date_format(COALESCE(p.period_month, g.period_month), 'yyyy-MM') AS year_month,
  concat('Q', COALESCE(p.quarter, g.quarter), ' ', COALESCE(p.year, g.year)) AS quarter_year,
  
  -- Employee metrics
  COALESCE(p.employee_count, 0) AS employee_count,
  
  -- Payroll metrics
  COALESCE(p.total_gross_salary, 0) AS payroll_gross_salary,
  COALESCE(p.total_bonus, 0) AS payroll_bonus,
  COALESCE(p.total_overtime, 0) AS payroll_overtime,
  COALESCE(p.total_commission, 0) AS payroll_commission,
  COALESCE(p.total_allowances, 0) AS payroll_allowances,
  COALESCE(p.total_compensation, 0) AS payroll_total_compensation,
  COALESCE(p.total_deductions, 0) AS payroll_total_deductions,
  COALESCE(p.total_tax_deduction, 0) AS payroll_tax,
  COALESCE(p.total_social_security, 0) AS payroll_social_security,
  COALESCE(p.total_health_insurance, 0) AS payroll_health_insurance,
  COALESCE(p.total_retirement, 0) AS payroll_retirement,
  COALESCE(p.total_other_deductions, 0) AS payroll_other_deductions,
  COALESCE(p.total_net_salary, 0) AS payroll_net_salary,
  COALESCE(p.avg_gross_salary, 0) AS avg_employee_salary,
  COALESCE(p.avg_total_compensation, 0) AS avg_employee_compensation,
  COALESCE(p.avg_overtime_ratio, 0) AS avg_overtime_ratio,
  
  -- GL Revenue metrics (now using credit_amount for gross revenue)
  COALESCE(g.total_revenue, 0) AS gl_revenue,
  
  -- GL Expense metrics by category (all positive)
  COALESCE(g.total_cogs, 0) AS gl_cogs,
  COALESCE(g.total_personnel_expense, 0) AS gl_personnel_expense,
  COALESCE(g.total_occupancy_expense, 0) AS gl_occupancy_expense,
  COALESCE(g.total_technology_expense, 0) AS gl_technology_expense,
  COALESCE(g.total_marketing_expense, 0) AS gl_marketing_expense,
  COALESCE(g.total_travel_expense, 0) AS gl_travel_expense,
  COALESCE(g.total_professional_services, 0) AS gl_professional_services,
  COALESCE(g.total_communication_expense, 0) AS gl_communication_expense,
  COALESCE(g.total_financial_expense, 0) AS gl_financial_expense,
  COALESCE(g.total_depreciation, 0) AS gl_depreciation,
  COALESCE(g.total_other_opex, 0) AS gl_other_opex,
  COALESCE(g.total_rnd_expense, 0) AS gl_rnd_expense,
  COALESCE(g.total_tax_expense, 0) AS gl_tax_expense,
  COALESCE(g.total_expenses, 0) AS gl_total_expenses,
  COALESCE(g.transaction_count, 0) AS gl_transaction_count,
  
  -- Calculated KPIs (now with credit_amount revenue)
  COALESCE(p.total_compensation, 0) + COALESCE(g.total_expenses, 0) AS total_departmental_cost,
  COALESCE(g.total_revenue, 0) - COALESCE(g.total_cogs, 0) AS gross_profit,
  COALESCE(g.total_revenue, 0) - COALESCE(g.total_cogs, 0) - COALESCE(g.total_expenses, 0) - COALESCE(p.total_compensation, 0) AS net_profit,
  CASE 
    WHEN COALESCE(g.total_revenue, 0) > 0 
    THEN ((COALESCE(g.total_revenue, 0) - COALESCE(g.total_cogs, 0)) / COALESCE(g.total_revenue, 0)) * 100 
    ELSE 0 
  END AS gross_margin_pct,
  CASE 
    WHEN COALESCE(g.total_revenue, 0) > 0 
    THEN ((COALESCE(g.total_revenue, 0) - COALESCE(g.total_cogs, 0) - COALESCE(g.total_expenses, 0) - COALESCE(p.total_compensation, 0)) / COALESCE(g.total_revenue, 0)) * 100 
    ELSE 0 
  END AS net_margin_pct,
  CASE 
    WHEN COALESCE(p.employee_count, 0) > 0 
    THEN COALESCE(g.total_revenue, 0) / COALESCE(p.employee_count, 0) 
    ELSE 0 
  END AS revenue_per_employee,
  CASE 
    WHEN COALESCE(p.employee_count, 0) > 0 
    THEN (COALESCE(p.total_compensation, 0) + COALESCE(g.total_expenses, 0)) / COALESCE(p.employee_count, 0) 
    ELSE 0 
  END AS cost_per_employee,
  
  current_timestamp() AS last_updated
FROM payroll_monthly p
FULL OUTER JOIN gl_monthly g 
  ON p.company_id = g.company_id 
  AND p.department_id = g.department_id 
  AND p.period_month = g.period_month;

--SELECT 'Datacube created with ' || COUNT(*) || ' rows' AS status FROM mini_project.gold.datacube_kpi;
select * from mini_project.gold.datacube_kpi